# Predict ergonomics demo

Use this notebook to inspect the `Predict` call surface in Jupyter and VS Code's notebook UI.

Things to try while this notebook is open:

- hover over `predict` and `inspect.signature(predict)`
- inspect the printed call signatures
- run the typed and untyped examples
- compare the positional calls with the keyword calls


In [ ]:
from __future__ import annotations

import inspect
import logging

import dspy


In [ ]:
class DummyPredictLM(dspy.LM):
    def __init__(self):
        super().__init__("dummy")
        self.calls = []

    def __call__(self, prompt=None, messages=None, **kwargs):
        self.calls.append({"prompt": prompt, "messages": messages, "kwargs": kwargs})
        return ["[[ ## answer ## ]]\nParis"]


class QA(dspy.Signature):
    """Answer a question using the supplied context."""

    question: str = dspy.InputField()
    context: list[str] = dspy.InputField()
    answer: str = dspy.OutputField()


class UntypedQuestion(dspy.Signature):
    """Answer a question with no explicit input type hint."""

    question = dspy.InputField()
    answer: str = dspy.OutputField()


class WithDefault(dspy.Signature):
    """Answer a question with optional context."""

    context: str = dspy.InputField(default="DEFAULT_CONTEXT")
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


In [ ]:
logging.basicConfig(level=logging.WARNING)
lm = DummyPredictLM()
dspy.configure(lm=lm)


## Typed signature

The typed input fields should appear in `inspect.signature(...)`.

In [ ]:
predict = dspy.Predict(QA)
inspect.signature(predict)


In [ ]:
predict("What is the capital of France?", ["Use world knowledge."])


In [ ]:
predict("What is the capital of France?", context=["Be concise."])


## Call-time signature override

Use `signature_override=` to try a temporary instruction set without mutating the predictor.

In [ ]:
concise = predict.signature.with_instructions("Answer in one short sentence.")
predict(
    question="Why is the sky blue?",
    context=["Use world knowledge."],
    signature_override=concise,
    config={"temperature": 0.2},
)


## Untyped signature

An untyped input should stay unannotated in the Python call signature.
This is the matching runtime behavior for PR 9313 as well: if the input was not explicitly typed, DSPy should not warn about a type mismatch.

In [ ]:
untyped = dspy.Predict(UntypedQuestion)
inspect.signature(untyped)


In [ ]:
untyped(question=123)


## Default-before-required edge case

Python's `inspect.Signature` does not allow a defaulted positional-or-keyword parameter to come before a required positional-or-keyword parameter.
The runtime still follows DSPy signature order, but the displayed call signature may switch later inputs to keyword-only so the displayed signature stays valid Python.

In [ ]:
with_default = dspy.Predict(WithDefault)
inspect.signature(with_default)


In [ ]:
with_default(question="What is the capital of France?")


## Last LM call payload

Useful for inspecting what was actually passed downstream.

In [ ]:
lm.calls[-1] if lm.calls else None
